# Hyperview — DOFA experiment

Uses the **DOFA** (Dynamic One-For-All) foundation model from
[torchgeo](https://torchgeo.readthedocs.io/en/latest/api/models.html#dofa).

Unlike the ConvNeXt experiments, **no PCA is needed**: DOFA accepts
all 150 raw spectral bands and uses their wavelength values (in µm) to
dynamically generate the patch-projection weights.

**Pipeline overview**
```
Raw 150-band patch  ──►  pad/crop 224×224  ──►  z-score normalise
      │                                               │
      └──────── wavelengths (µm) ──────────────────►DOFA ViT-B/16
                                                      │
                                        CLS feature (768-d)
                                                      │
                                     + log(valid_pixels) scalar
                                                      │
                                         MLP regressor → 4 outputs
```


## 1 · Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

for d in ['/content/train_data', '/content/test_data']:
    os.makedirs(d, exist_ok=True)

!tar -xzf /content/drive/MyDrive/train.tar.gz -C /content/train_data
!tar -xzf /content/drive/MyDrive/test.tar.gz  -C /content/test_data
!cp /content/drive/MyDrive/train_gt.csv    /content/train_gt.csv
!cp /content/drive/MyDrive/wavelengths.csv /content/wavelengths.csv
print('Data ready.')


In [ ]:
# torchgeo >= 0.6 ships DOFA; wandb, joblib, tqdm for training utilities
!pip install -q 'torchgeo>=0.6' wandb joblib tqdm


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B key loaded from Colab Secrets.')
except Exception:
    # os.environ['WANDB_API_KEY'] = 'paste-your-key-here'
    print('WANDB_API_KEY not set — wandb.login() will prompt interactively.')


## 2 · Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import wandb, joblib
from tqdm import tqdm

# DOFA from torchgeo
from torchgeo.models import DOFA, DOFABase16_Weights, DOFALarge16_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## 3 · Data loading


In [ ]:
def load_data(directory: str):
    """Load all .npz patches.  Returns list of (img_tensor, mask) tuples.
    img_tensor : (150, H, W) float32  — raw reflectance, invalid pixels kept as-is
    mask       : (150, H, W) bool     — True = valid pixel
    """
    data = []
    all_files = np.array(sorted(
        glob(os.path.join(directory, '*.npz')),
        key=lambda x: int(os.path.basename(x).replace('.npz', ''))
    ))
    for f in all_files:
        with np.load(f) as npz:
            raw = np.ma.MaskedArray(data=npz['data'], mask=npz['mask'])
            img  = torch.as_tensor(raw.data, dtype=torch.float32)
            mask = ~torch.as_tensor(raw.mask)   # True = valid
        data.append((img, mask))
    return data


def load_gt(file_path: str) -> np.ndarray:
    return pd.read_csv(file_path)[['P', 'K', 'Mg', 'pH']].values


X_all = load_data('/content/train_data/train')
y_all = load_gt('/content/train_gt.csv')
print(f'Patches: {len(X_all)}  |  Spectral bands: {X_all[0][0].shape[0]}')
print(f'Labels:  {y_all.shape}')


In [ ]:
indices = np.arange(len(X_all))

X_tmp, X_test, y_tmp, y_test, idx_tmp, idx_test = train_test_split(
    X_all, y_all, indices, test_size=0.20, random_state=93
)
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_tmp, y_tmp, idx_tmp, test_size=0.25, random_state=93
)
print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')


## 4 · Wavelengths

DOFA needs the centre wavelength (in **µm**) of every spectral band so it can
dynamically generate the patch-projection weights.
The `wavelengths.csv` file stores them in **nm** → divide by 1 000.


In [ ]:
wavelength_df = pd.read_csv('/content/wavelengths.csv')
print('wavelengths.csv columns:', wavelength_df.columns.tolist())
print(wavelength_df.head())


In [ ]:
# ── adapt the column name below if your CSV uses a different header ──
# Common names: 'Wavelength', 'wavelength', 'WL', 'nm'
WL_COL = wavelength_df.columns[0]          # take the first column
wavelengths_nm = wavelength_df[WL_COL].values.astype(float)   # (150,) in nm
wavelengths_um = (wavelengths_nm / 1000.0).tolist()            # µm — required by DOFA

print(f'Number of bands : {len(wavelengths_um)}')
print(f'Spectral range  : {wavelengths_nm.min():.1f} – {wavelengths_nm.max():.1f} nm')
print(f'                  {min(wavelengths_um):.4f} – {max(wavelengths_um):.4f} µm')

plt.figure(figsize=(10, 2))
plt.hlines(0, wavelengths_nm.min(), wavelengths_nm.max(), color='steelblue', lw=2)
plt.scatter(wavelengths_nm, np.zeros_like(wavelengths_nm), s=15, color='steelblue')
plt.xlabel('Wavelength (nm)'); plt.title('Spectral bands covered by the sensor')
plt.yticks([]); plt.tight_layout(); plt.show()


## 5 · Per-channel normalisation statistics

We compute mean and std over **valid pixels only** from the training split.
This replaces the PCA step used in ConvNeXt experiments — DOFA receives
the raw 150-band data normalised band-by-band.

> **Note:** computing stats for 150 channels takes ~2–4 min on CPU.


In [ ]:
import torch.nn.functional as F

PAD_SIZE = (224, 224)

def pad_to_size(x, mask, size=PAD_SIZE):
    _, h, w = x.shape
    th, tw  = size
    if h > th:
        sh = (h - th) // 2
        x, mask = x[:, sh:sh+th, :], mask[:, sh:sh+th, :]; h = th
    if w > tw:
        sw = (w - tw) // 2
        x, mask = x[:, :, sw:sw+tw], mask[:, :, sw:sw+tw]; w = tw
    ph, pw = th - h, tw - w
    pt, pl = ph // 2, pw // 2
    x    = F.pad(x,    (pl, pw-pl, pt, ph-pt), value=0)
    mask = F.pad(mask, (pl, pw-pl, pt, ph-pt), value=0)
    return x, mask


def calculate_global_stats(data_list, size=PAD_SIZE):
    """Per-channel mean/std over valid pixels from all training patches."""
    print('Computing per-channel stats (150 bands) — may take a few minutes …')
    n_ch = data_list[0][0].shape[0]
    sums   = torch.zeros(n_ch)
    sums_sq= torch.zeros(n_ch)
    counts = torch.zeros(n_ch)
    for x, mask in tqdm(data_list, desc='Global stats'):
        xp, mp = pad_to_size(x, mask, size)
        valid  = mp[0].bool()                         # (H, W) spatial mask
        vals   = xp[:, valid]                         # (C, N_valid)
        sums   += vals.sum(dim=1)
        sums_sq+= (vals ** 2).sum(dim=1)
        counts += valid.sum().float()
    means = sums / counts
    stds  = (sums_sq / counts - means ** 2).sqrt().clamp(min=1e-6)
    print(f'Done.  Mean range: [{means.min():.4f}, {means.max():.4f}]')
    print(f'       Std  range: [{stds.min():.4f},  {stds.max():.4f}]')
    return means, stds


global_means, global_stds = calculate_global_stats(X_train)
torch.save({'means': global_means, 'stds': global_stds}, 'global_stats_dofa.pt')
print('global_stats_dofa.pt saved')


## 6 · Label scaling


In [ ]:
scaler_y = StandardScaler()
y_train_sc = scaler_y.fit_transform(y_train)
y_val_sc   = scaler_y.transform(y_val)
y_test_sc  = scaler_y.transform(y_test)
joblib.dump(scaler_y, 'scaler_labels_dofa.pkl')
print('scaler_labels_dofa.pkl saved')


## 7 · Dataset

Same design as the ConvNeXt experiments: pad/crop to 224×224, z-score
normalisation on valid pixels, spatial augmentations, invalid pixels zeroed
out.  The dataset returns **(x, num_valid_pixels, y)** or
**(x, num_valid_pixels)** for the submission set.


In [ ]:
class RandomRotate90:
    def __call__(self, x):
        k = torch.randint(0, 4, (1,)).item()
        return torch.rot90(x, k, dims=(-2, -1))


class RandomSpectralDrop:
    """Zero-out random spectral channels — prevents over-reliance on specific bands."""
    def __init__(self, drop_prob: float = 0.05):
        self.drop_prob = drop_prob
    def __call__(self, x):
        keep = torch.bernoulli(torch.ones(x.shape[0]) * (1 - self.drop_prob))
        return x * keep.view(-1, 1, 1)


class NPZDataset(Dataset):
    """Dataset for raw 150-band hyperspectral patches (no PCA).

    tensor_list : list of (img_tensor, mask) tuples
                  img_tensor : (150, H, W) float32
                  mask       : (150, H, W) bool, True = valid
    labels      : np.ndarray (N, 4) or None (submission)
    augment     : apply spatial + spectral augmentations
    size        : target spatial size after pad/crop
    """
    def __init__(self, tensor_list, labels=None, augment=True,
                 size=PAD_SIZE, global_means=None, global_stds=None):
        self.tensor_list  = tensor_list
        self.labels       = labels
        self.augment      = augment
        self.size         = size
        self.global_means = global_means
        self.global_stds  = global_stds
        self.transform_aug = v2.Compose([
            v2.RandomResizedCrop(size=self.size, scale=(0.8, 1.0)),
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.5),
            RandomRotate90(),
            RandomSpectralDrop(drop_prob=0.05),
        ])

    def __len__(self):
        return len(self.tensor_list)

    def _pad(self, x, mask):
        return pad_to_size(x, mask, self.size)

    def _normalize(self, x, mask):
        if self.global_means is not None:
            x = (x - self.global_means.view(-1,1,1)) / \
                (self.global_stds.view(-1,1,1) + 1e-6)
        return x * mask.float()   # zero-out invalid pixels

    def __getitem__(self, idx):
        x, mask = self.tensor_list[idx]
        x, mask = self._pad(x, mask)
        num_valid = mask[0].sum().float()
        x = self._normalize(x, mask)
        if self.augment and self.labels is not None:
            x = self.transform_aug(x)
            x = x * mask.float()   # re-zero after spatial aug
        if self.labels is not None:
            return x, num_valid, torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, num_valid


In [ ]:
BATCH_SIZE  = 16   # ViT-B/16 is memory-hungry; reduce to 8 if OOM on Colab
NUM_WORKERS = 2

train_ds = NPZDataset(X_train, y_train_sc, augment=True,
                      global_means=global_means, global_stds=global_stds)
val_ds   = NPZDataset(X_val,   y_val_sc,   augment=False,
                      global_means=global_means, global_stds=global_stds)
test_ds  = NPZDataset(X_test,  y_test_sc,  augment=False,
                      global_means=global_means, global_stds=global_stds)

kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **kw)

# Quick sanity check
xb, nv, yb = next(iter(train_loader))
print('Batch shapes — x:', xb.shape, ' nv:', nv.shape, ' y:', yb.shape)


## 8 · DOFA model

### Architecture
```
DOFA ViT-B/16  (embed_dim=768, depth=12, heads=12)
  └─ wavelength-conditioned patch projection:  150×16×16 → 768
  └─ 12 × Transformer blocks
  └─ CLS token  →  (B, 768)

Regressor head:
  Linear(769→256) → GELU → Dropout(0.3)
  Linear(256→64)  → GELU → Dropout(0.15)
  Linear(64→4)
```

`num_valid_pixels` is log-scaled and concatenated to the backbone features
so the head can adjust confidence based on patch size — important for the
small irregular patches that dominate prediction errors in HyperView.

### Pretrained weights
Set `DOFA_VARIANT = 'base'` (768-d) or `'large'` (1024-d).
Both use MAE pretraining on the EO-1M Earth observation dataset.


In [ ]:
DOFA_VARIANT = 'base'   # 'base' (768-d) or 'large' (1024-d)


class DOFARegressor(nn.Module):
    """DOFA backbone + MLP regression head.

    Args:
        wavelengths_um : list of floats — band centre wavelengths in µm
                         (convert nm → µm before passing here)
        n_outputs      : number of regression targets (4: P, K, Mg, pH)
        dropout        : dropout rate in the regression head
        variant        : 'base' (ViT-B, 768-d) or 'large' (ViT-L, 1024-d)
        pretrained     : load DOFA MAE pretrained weights from torchgeo
    """

    _CFG = {
        'base':  dict(embed_dim=768,  depth=12, num_heads=12,
                      weights_cls=DOFABase16_Weights),
        'large': dict(embed_dim=1024, depth=24, num_heads=16,
                      weights_cls=DOFALarge16_Weights),
    }

    def __init__(
        self,
        wavelengths_um: list,
        n_outputs: int  = 4,
        dropout: float  = 0.3,
        variant: str    = 'base',
        pretrained: bool = True,
    ):
        super().__init__()
        cfg = self._CFG[variant]
        embed_dim = cfg['embed_dim']

        # Wavelengths stored as a buffer so they move with .to(device) and
        # are saved with the state_dict
        self.register_buffer(
            'wavelengths',
            torch.tensor(wavelengths_um, dtype=torch.float32)
        )

        # ── DOFA backbone ────────────────────────────────────────────────
        self.backbone = DOFA(
            image_size  = 224,
            patch_size  = 16,
            embed_dim   = cfg['embed_dim'],
            depth       = cfg['depth'],
            num_heads   = cfg['num_heads'],
            mlp_ratio   = 4.0,
            num_classes = 0,    # return CLS features, no classification head
        )

        if pretrained:
            weights = cfg['weights_cls'].DOFA_MAE
            sd = weights.get_state_dict(progress=True)
            missing, unexpected = self.backbone.load_state_dict(sd, strict=False)
            print(f'[DOFA-{variant}] pretrained weights loaded.')
            if missing:
                print(f'  Missing keys    ({len(missing)}): {missing[:5]}')
            if unexpected:
                print(f'  Unexpected keys ({len(unexpected)}): {unexpected[:5]}')
        else:
            print(f'[DOFA-{variant}] random initialisation.')

        # ── regression head ──────────────────────────────────────────────
        # +1 for log(valid_pixel_count) patch-size feature
        self.regressor = nn.Sequential(
            nn.Linear(embed_dim + 1, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(64, n_outputs),
        )

    def forward(self, x: torch.Tensor, num_valid_pixels: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x                : (B, 150, 224, 224) normalised reflectance
            num_valid_pixels : (B,) count of valid (non-padded) pixels
        Returns:
            (B, n_outputs)
        """
        wl = self.wavelengths.tolist()                                  # list[float]
        features  = self.backbone(x, wl)                               # (B, embed_dim)
        size_feat = torch.log1p(num_valid_pixels.float()) / 10.0       # (B,)
        features  = torch.cat([features, size_feat.unsqueeze(1)], 1)  # (B, embed_dim+1)
        return self.regressor(features)


## 9 · Sanity-check: overfit on a single batch

Overfitting loss should reach < 0.01 within ~200 epochs if the model and
data pipeline are wired up correctly.


In [ ]:
model_test = DOFARegressor(
    wavelengths_um = wavelengths_um,
    variant        = DOFA_VARIANT,
    pretrained     = True,
    dropout        = 0.0,   # disable dropout for overfit test
).to(device)

x_s, nv_s, y_s = xb.to(device), nv.to(device), yb.to(device)  # reuse first batch

opt_s = optim.AdamW(model_test.parameters(), lr=1e-4, weight_decay=0)
crit  = nn.MSELoss()

print(f'Overfitting on batch of {x_s.shape[0]} samples for 200 epochs …')
for ep in range(200):
    model_test.train()
    opt_s.zero_grad()
    loss = crit(model_test(x_s, nv_s), y_s)
    loss.backward()
    opt_s.step()
    if (ep + 1) % 50 == 0 or ep == 0:
        print(f'  Epoch {ep+1:4d}/200   loss={loss.item():.6f}')

del model_test   # free GPU memory
torch.cuda.empty_cache()


## 10 · Training loop


In [ ]:
def train_with_early_stopping(
    model, train_loader, val_loader, optimizer, scheduler, criterion, device,
    epochs=90, warmup_epochs=10, patience=13,
    save_path='best_dofa_model.pth', max_grad_norm=1.0, wandb_run=None
):
    """Linear LR warm-up · ReduceLROnPlateau · gradient clipping · early stopping."""
    initial_lrs   = [pg['lr'] for pg in optimizer.param_groups]
    best_val_loss = float('inf')
    ctr           = 0

    for epoch in range(epochs):
        # ── warm-up ──
        if epoch < warmup_epochs:
            wf = (epoch + 1) / float(warmup_epochs)
            for i, pg in enumerate(optimizer.param_groups):
                pg['lr'] = initial_lrs[i] * wf

        # ── train ──
        model.train()
        t_sum, t_n = 0.0, 0
        loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False)
        for xb, nv, yb in loop:
            xb, nv, yb = xb.to(device), nv.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb, nv), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            t_sum += loss.item() * xb.size(0); t_n += xb.size(0)
            loop.set_postfix(loss=f'{loss.item():.4f}')
        train_loss = t_sum / max(1, t_n)

        # ── validate ──
        model.eval()
        v_sum, v_n = 0.0, 0
        with torch.no_grad():
            for xv, nv, yv in val_loader:
                xv, nv, yv = xv.to(device), nv.to(device), yv.to(device)
                v_sum += criterion(model(xv, nv), yv).item() * xv.size(0)
                v_n   += xv.size(0)
        val_loss = v_sum / max(1, v_n)

        if epoch >= warmup_epochs:
            scheduler.step(val_loss)

        lrs = [pg['lr'] for pg in optimizer.param_groups]
        print(f'Epoch {epoch+1:3d}/{epochs}  '
              f'Train: {train_loss:.4f}  Val: {val_loss:.4f}  '
              f'LR backbone: {lrs[0]:.2e}  LR head: {lrs[1]:.2e}')

        if wandb_run:
            wandb_run.log({'train_loss': train_loss, 'val_loss': val_loss,
                           'epoch': epoch + 1,
                           'lr_backbone': lrs[0], 'lr_head': lrs[1]})

        if val_loss < best_val_loss:
            best_val_loss = val_loss; ctr = 0
            torch.save(model.state_dict(), save_path)
            print(f'  ✓ Saved best model (val_loss={best_val_loss:.4f})')
        else:
            ctr += 1
            if ctr >= patience:
                print(f'Early stopping at epoch {epoch+1} (best val: {best_val_loss:.4f})')
                break

    model.load_state_dict(torch.load(save_path, map_location=device))
    print(f'Best model loaded (val loss: {best_val_loss:.4f})')
    return model


## 11 · Build model, optimiser, and run training

Two param groups with differential learning rates:
- **Backbone** (pretrained DOFA): `lr = 1e-5`  — small LR to fine-tune gently
- **Regressor head** (random init): `lr = 1e-4` — larger LR to learn fast


In [ ]:
EPOCHS        = 90
WARMUP_EPOCHS = 10
PATIENCE      = 13
WEIGHT_DECAY  = 1e-4
LR_BACKBONE   = 1e-5
LR_HEAD       = 1e-4

# ── model ──
model = DOFARegressor(
    wavelengths_um = wavelengths_um,
    variant        = DOFA_VARIANT,
    pretrained     = True,
    dropout        = 0.3,
).to(device)

# ── optimizer ──
optimizer = optim.AdamW([
    {'params': model.backbone.parameters(),  'lr': LR_BACKBONE},
    {'params': model.regressor.parameters(), 'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)

# ── loss ──
criterion = nn.MSELoss()

# ── scheduler ──
scheduler = ReduceLROnPlateau(
    optimizer, mode='min', factor=0.3, patience=6, min_lr=1e-6
)

# ── W&B ──
wandb.init(
    project = 'hyperview_challenge',
    name    = f'dofa_{DOFA_VARIANT}_raw150',
    config  = {
        'model':          f'DOFA-{DOFA_VARIANT}',
        'spectral_mode':  'raw_150_bands',
        'pretrained':     True,
        'lr_backbone':    LR_BACKBONE,
        'lr_head':        LR_HEAD,
        'weight_decay':   WEIGHT_DECAY,
        'batch_size':     BATCH_SIZE,
        'epochs':         EPOCHS,
        'warmup_epochs':  WARMUP_EPOCHS,
        'patience':       PATIENCE,
        'pad_size':       PAD_SIZE,
        'criterion':      'MSELoss',
        'scheduler':      'ReduceLROnPlateau(factor=0.3, patience=6)',
    }
)

# ── train ──
train_with_early_stopping(
    model         = model,
    train_loader  = train_loader,
    val_loader    = val_loader,
    optimizer     = optimizer,
    scheduler     = scheduler,
    criterion     = criterion,
    device        = device,
    epochs        = EPOCHS,
    warmup_epochs = WARMUP_EPOCHS,
    patience      = PATIENCE,
    save_path     = 'best_dofa_model.pth',
    max_grad_norm = 1.0,
    wandb_run     = wandb.run,
)
wandb.finish()


## 12 · Evaluation vs. baseline


In [ ]:
class BaselineRegressor:
    """Always predicts the training-set mean — competition baseline."""
    def fit(self, X, y):
        self.mean = np.mean(y, axis=0)
        self.n    = y.shape[1]
        return self
    def predict(self, X):
        return np.full((len(X), self.n), self.mean)


class SpectralCurveFiltering:
    """Mean spectrum over all pixels (H, W) → (C,)."""
    def __call__(self, x):
        return x.mean(axis=(1, 2))


def evaluate_dl_model(model, loader, scaler_y, device):
    model.eval()
    preds_sc, true_sc = [], []
    with torch.no_grad():
        for xb, nv, yb in loader:
            preds_sc.append(model(xb.to(device), nv.to(device)).cpu().numpy())
            true_sc.append(yb.numpy())
    y_pred = scaler_y.inverse_transform(np.vstack(preds_sc))
    y_true = scaler_y.inverse_transform(np.vstack(true_sc))
    mse    = np.mean((y_true - y_pred) ** 2, axis=0)
    return y_pred, y_true, mse


def calculate_and_print_results(model_mse, baseline_mse, y_true, y_pred, bl_pred):
    score  = np.mean(model_mse / baseline_mse)
    names  = ['P', 'K', 'Mg', 'pH']
    print('Per-target comparison:')
    for i, n in enumerate(names):
        print(f'  {n}: Model={model_mse[i]:.4f}  '
              f'Baseline={baseline_mse[i]:.4f}  '
              f'Norm={model_mse[i]/baseline_mse[i]:.4f}')
    print(f'\nChallenge score (lower is better): {score:.4f}')

    plt.figure(figsize=(15, 5))
    for i, n in enumerate(names):
        plt.subplot(1, 4, i + 1)
        plt.scatter(y_true[:, i], y_pred[:, i],  alpha=0.7, label='DOFA')
        plt.scatter(y_true[:, i], bl_pred[:, i], alpha=0.7, label='Baseline', marker='x')
        lo = min(y_true[:, i].min(), y_pred[:, i].min())
        hi = max(y_true[:, i].max(), y_pred[:, i].max())
        plt.plot([lo, hi], [lo, hi], 'r--')
        plt.title(n); plt.xlabel('True'); plt.ylabel('Predicted')
        plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.show()
    return score


In [ ]:
# ── DL model evaluation ──
model.load_state_dict(torch.load('best_dofa_model.pth', map_location=device))
dl_preds, y_test_true, dl_mse = evaluate_dl_model(model, test_loader, scaler_y, device)

# ── Baseline (mean-prediction) ──
filtering       = SpectralCurveFiltering()
X_test_raw      = [X_all[i][0] for i in idx_test]
X_test_filt     = np.array([filtering(c.numpy()) for c in X_test_raw])

combined_idx    = idx_train.tolist() + idx_val.tolist()
X_tv_raw        = [X_all[i][0] for i in combined_idx]
X_tv_filt       = np.array([filtering(c.numpy()) for c in X_tv_raw])

baseline = BaselineRegressor().fit(X_tv_filt, y_all[combined_idx])
bl_preds = baseline.predict(X_test_filt)
bl_mse   = np.mean((y_test_true - bl_preds) ** 2, axis=0)

score = calculate_and_print_results(dl_mse, bl_mse, y_test_true, dl_preds, bl_preds)


In [ ]:
names  = ['P', 'K', 'Mg', 'pH']
x_pos  = np.arange(len(names))
width  = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
r1 = ax.bar(x_pos - width/2, dl_mse,  width, label='DOFA model')
r2 = ax.bar(x_pos + width/2, bl_mse,  width, label='Baseline')
for r in [r1, r2]:
    for rect in r:
        h = rect.get_height()
        ax.annotate(f'{h:.2f}',
                    xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9)
ax.set_xticks(x_pos); ax.set_xticklabels(names)
ax.set_ylabel('MSE'); ax.set_title('DOFA vs. Baseline MSE per target')
ax.legend(); fig.tight_layout(); plt.show()
print(f'Challenge score: {score:.4f}')


## 13 · Submission inference

Run this section **after** training to produce `submission.csv`.


In [ ]:
import gc
del train_ds, val_ds, test_ds, train_loader, val_loader, test_loader
gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ── Load saved artefacts ──
stats        = torch.load('global_stats_dofa.pt')
global_means = stats['means']
global_stds  = stats['stds']
scaler_y     = joblib.load('scaler_labels_dofa.pkl')

# ── Load submission test patches ──
X_sub = load_data('/content/test_data/test')
sub_ds     = NPZDataset(X_sub, augment=False,
                        global_means=global_means, global_stds=global_stds)
sub_loader = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

# ── Reload best model ──
model = DOFARegressor(
    wavelengths_um = wavelengths_um,
    variant        = DOFA_VARIANT,
    pretrained     = False,    # weights loaded from checkpoint below
).to(device)
model.load_state_dict(torch.load('best_dofa_model.pth', map_location=device))
model.eval()

# ── Inference ──
preds_sc = []
with torch.no_grad():
    for xb, nv in sub_loader:
        preds_sc.append(model(xb.to(device), nv.to(device)).cpu().numpy())

y_pred = scaler_y.inverse_transform(np.vstack(preds_sc))
import pandas as pd
pd.DataFrame(y_pred, columns=['P', 'K', 'Mg', 'pH']).to_csv(
    'submission_dofa.csv', index_label='sample_index'
)
print('submission_dofa.csv saved')
